# Cleaning Data
**Track:** Data Analytics — Level 1, Task 3
**Objective:** Take a deliberately messy dataset and systematically transform it into a clean, analysis-ready dataset, documenting every decision.

Dataset: `messy_customer_data.csv` — a synthetic customer dataset with intentionally injected nulls, duplicates, inconsistent text formatting, mixed date formats, wrong data types, and outliers.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("messy_customer_data.csv")
df.head()

,CustomerID,Name,Age,Gender,City,JoinDate,Salary,PurchaseAmount
0,1,Kavya Nair,43.0,MALE,Delhi,07/31/2022,21155.99,919.05
1,2,Arjun Patel,57.0,MALE,delhi,03/09/2023,47170.39,1205.19
2,3,Aarav Singh,24.0,Female,Bangalore,2024-04-04,70904.28,882.69
3,4,Aditya Patel,27.0,MALE,bangalore,12-02-2024,39942.20,428.74
4,5,Ananya Reddy,26.0,Female,Mumbai,2022-09-15,54261.24,2853.45


## 1. Data Quality Report (before cleaning)

In [2]:
report_before = {
    "rows": len(df),
    "duplicate_rows": df.duplicated().sum(),
    "nulls_per_column": df.isnull().sum().to_dict(),
    "dtypes": df.dtypes.astype(str).to_dict()
}
for k, v in report_before.items():
    print(k, ":", v)

rows : 520
duplicate_rows : 20
nulls_per_column : {'CustomerID': 0, 'Name': 0, 'Age': 21, 'Gender': 15, 'City': 10, 'JoinDate': 0, 'Salary': 25, 'PurchaseAmount': 17}
dtypes : {'CustomerID': 'int64', 'Name': 'str', 'Age': 'float64', 'Gender': 'str', 'City': 'str', 'JoinDate': 'str', 'Salary': 'float64', 'PurchaseAmount': 'float64'}


**Observation:** `Age` is stored as an `object` (mixed string/number), `JoinDate` is a plain string with multiple formats,
`Gender` and `City` have inconsistent capitalisation/spacing, and there are duplicate rows plus a batch of nulls across several columns.

## 2. Handle Missing Data
A different strategy is justified for each column based on its nature.

In [3]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Salary"] = df["Salary"].fillna(df["Salary"].median())

df["PurchaseAmount"] = df["PurchaseAmount"].fillna(df["PurchaseAmount"].median())

df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["City"] = df["City"].fillna(df["City"].mode()[0])

df.isnull().sum()

CustomerID        0
Name              0
Age               0
Gender            0
City              0
JoinDate          0
Salary            0
PurchaseAmount    0
dtype: int64

**Justification:**
- `Age`, `Salary`, `PurchaseAmount` (numeric, skewed by outliers) → **median imputation**, which is robust to outliers unlike the mean.
- `Gender`, `City` (categorical) → **mode imputation**, filling with the most frequent category, since these are unordered labels with no natural "average".
- No column had enough missingness (>5%) to justify row deletion, which would lose otherwise-valid data.

## 3. Remove Duplicates

In [4]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} duplicate rows ({before} -> {after})")

Removed 20 duplicate rows (520 -> 500)


## 4. Standardise Inconsistent Formatting

In [5]:
df["Gender"] = df["Gender"].str.strip().str.upper().map({
    "MALE": "Male", "M": "Male", "FEMALE": "Female", "F": "Female"
}).fillna(df["Gender"])

df["City"] = df["City"].str.strip().str.title()

df["JoinDate"] = pd.to_datetime(df["JoinDate"], format="mixed", dayfirst=False, errors="coerce")

print(df["Gender"].unique())
print(df["City"].unique())
print(df["JoinDate"].head())

<StringArray>
['Male', 'Female']
Length: 2, dtype: str
<StringArray>
['Delhi', 'Bangalore', 'Mumbai', 'Hyderabad', 'Pune', 'Chennai']
Length: 6, dtype: str
0   2022-07-31
1   2023-03-09
2   2024-04-04
3   2024-12-02
4   2022-09-15
Name: JoinDate, dtype: datetime64[us]


**Observation:** `Gender` now has exactly 2 clean categories, `City` has consistent Title Case naming (no more "mumbai" vs "MUMBAI" vs "Mumbai" being treated as different values), and `JoinDate` is a proper `datetime` column regardless of its original format (`YYYY-MM-DD`, `DD/MM/YYYY`, etc.).

## 5. Outlier Detection (IQR method)

In [6]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

for col in ["Salary", "PurchaseAmount"]:
    low, high = iqr_bounds(df[col])
    n_outliers = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds=({low:.0f}, {high:.0f}), outliers={n_outliers}")

Salary: bounds=(6382, 103073), outliers=8
PurchaseAmount: bounds=(-3490, 8350), outliers=29


In [7]:
# Decision: cap Salary outliers (extreme but plausible high earners) rather than drop them
low, high = iqr_bounds(df["Salary"])
df["Salary"] = df["Salary"].clip(lower=low, upper=high)

# Decision: PurchaseAmount cannot be negative -- these are data-entry errors, so we take the absolute value
df["PurchaseAmount"] = df["PurchaseAmount"].abs()

print("Salary range after capping:", df["Salary"].min(), "-", df["Salary"].max())
print("Min PurchaseAmount after fix:", df["PurchaseAmount"].min())

Salary range after capping: 12000.0 - 103072.83749999997
Min PurchaseAmount after fix: 15.25


**Decision & justification:** `Salary` outliers were **capped** (Winsorized) at the IQR bounds — they're extreme but not impossible, so capping preserves the row while limiting distortion.
`PurchaseAmount` negative values were **corrected via absolute value** since a negative purchase amount is a data-entry sign error, not a real outlier — the transaction itself is still valid data.

## 6. Data Type Correction

In [8]:
df["CustomerID"] = df["CustomerID"].astype(str)
df["Age"] = df["Age"].astype(int)
df["Salary"] = df["Salary"].round(2).astype(float)
df["PurchaseAmount"] = df["PurchaseAmount"].round(2).astype(float)
# JoinDate already converted to datetime above

df.dtypes

CustomerID                   str
Name                         str
Age                        int64
Gender                       str
City                         str
JoinDate          datetime64[us]
Salary                   float64
PurchaseAmount           float64
dtype: object

## 7. Before vs. After Summary

In [9]:
report_after = {
    "rows": len(df),
    "duplicate_rows": df.duplicated().sum(),
    "nulls_total": df.isnull().sum().sum(),
}

summary = pd.DataFrame({
    "Metric": ["Row count", "Duplicate rows", "Total nulls", "Age dtype", "JoinDate dtype"],
    "Before": [report_before["rows"], report_before["duplicate_rows"], sum(report_before["nulls_per_column"].values()), "object", "object"],
    "After": [report_after["rows"], report_after["duplicate_rows"], report_after["nulls_total"], str(df["Age"].dtype), str(df["JoinDate"].dtype)]
})
summary

,Metric,Before,After
0,Row count,520,500
1,Duplicate rows,20,0
2,Total nulls,88,0
3,Age dtype,object,int64
4,JoinDate dtype,object,datetime64[us]


**Observation:** Row count dropped only due to duplicate removal, nulls went from several dozen to zero, and both `Age` and `JoinDate` now have correct, analysis-ready dtypes.

## 8. Save Cleaned Dataset

In [10]:
df.to_csv("cleaned_customer_data.csv", index=False)
print("Saved cleaned_customer_data.csv with shape", df.shape)
df.head()

Saved cleaned_customer_data.csv with shape (500, 8)


,CustomerID,Name,Age,Gender,City,JoinDate,Salary,PurchaseAmount
0,1,Kavya Nair,43,Male,Delhi,2022-07-31,21155.99,919.05
1,2,Arjun Patel,57,Male,Delhi,2023-03-09,47170.39,1205.19
2,3,Aarav Singh,24,Female,Bangalore,2024-04-04,70904.28,882.69
3,4,Aditya Patel,27,Male,Bangalore,2024-12-02,39942.20,428.74
4,5,Ananya Reddy,26,Female,Mumbai,2022-09-15,54261.24,2853.45


## Conclusion
This notebook took a realistically messy customer dataset and, through a documented sequence of decisions (median/mode imputation, duplicate removal, text standardisation, IQR-based outlier handling, and dtype correction), produced a clean CSV ready for downstream analysis or modelling.